# 🚀 Google Colab YOLO P2-Head 高解析度訓練筆記本
本筆記本專門用於在 Google Colab 上使用 GPU (T4 / A100 / L4) 進行 YOLO P2-Head (4-Head 小物件檢測) 與 Copy-Paste 數據增強高解析度訓練。

### Step 1: 檢查 GPU 狀態與安裝必備套件 (請確認選單已切換為 T4 GPU)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
else:
    print('⚠️ 警告：當前未開啟 GPU！請點擊選單 [執行階段] -> [變更執行階段類型] -> 選擇 [T4 GPU]')

!nvidia-smi
!pip install -q ultralytics opencv-python pyyaml

### Step 2: 解壓專案壓縮包 `project_colab.zip`

In [ ]:
# 請在左側「檔案 📁」區塊上傳本地的 project_colab.zip，然後執行此 Cell 解壓：
!unzip -o -q /content/project_colab.zip -d /content/
print("✅ 專案與資料集解壓完成！")

### Step 3: 動態搜尋資料集與啟動 P2-Head 高解析度 (1024x1024) 訓練

In [ ]:
from ultralytics import YOLO
import os, glob, torch

# 自動搜尋 data.yaml 路徑
data_yaml = None
for root, _, files in os.walk("/content"):
    if "data.yaml" in files:
        data_yaml = os.path.join(root, "data.yaml")
        break

if not data_yaml:
    raise FileNotFoundError("❌ 找不到 data.yaml！請確認是否已上傳 project_colab.zip 並執行 Step 2 解壓！")

print(f"✅ 找到資料集配置檔: {data_yaml}")

dataset_dir = os.path.dirname(data_yaml)
os.chdir(dataset_dir)
print(f"Current working directory: {os.getcwd()}")

device_setting = 0 if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device_setting}")

# 載入 P2-Head (4 Detection Heads: P2/P3/P4/P5) 小物件偵測架構
model = YOLO("yolov8s-p2.yaml")

# 高解析度 1024x1024 GPU 訓練
results = model.train(
    data=data_yaml,
    epochs=50,
    imgsz=1024,
    batch=16,
    cache=True,
    device=device_setting,
    project="runs/detect",
    name="colab_yolo_p2_highres"
)
print("🎉 訓練成功完成！")

### Step 4: 打包並下載 Colab 上的全部訓練結果與模型檔

In [ ]:
from google.colab import files
import shutil, os

# 1. 打包 runs 目錄 (包含 best.pt, last.pt, 評估圖表 results.png, confusion_matrix.png 等)
shutil.make_archive('/content/colab_training_results', 'zip', '/content/runs')
print("📦 已將訓練結果與模型壓縮為 colab_training_results.zip")

# 2. 發起瀏覽器自動下載
files.download('/content/colab_training_results.zip')
print("⬇️ colab_training_results.zip 已發起下載！")